In [4]:
%reload_ext autoreload
%autoreload 2

from tqdm import tqdm
from experiment_utils import load_data, run_experiment
from joblib import Parallel, delayed
import pandas as pd

In [7]:
X_train, X_val, y_train, y_val = load_data(dataset = 'bank', task = 'classification')

Ks = [5,10, 15]
alphas = [0.1,0.5,1,5]
seeds = range(40,50)

tasks = [(K, alpha, seed) for K in Ks for alpha  in alphas for seed in seeds]

if __name__ == '__main__':
    results = Parallel(n_jobs= -1)(
       delayed(run_experiment)(*t, X_train, y_train, X_val, y_val, task = 'classification', scenario = 1) 
       for t in tqdm(tasks, desc = 'Overall Experiment Progress')
    )
    bank_df_classification = pd.DataFrame(results)

Overall Experiment Progress: 100%|██████████| 120/120 [1:13:50<00:00, 36.92s/it]


In [12]:
print(bank_df_classification[['alpha','ifls_r2']])

     alpha   ifls_r2
0      0.1  0.081139
1      0.1  0.024328
2      0.1  0.013060
3      0.1  0.019754
4      0.1  0.202921
..     ...       ...
115    5.0  0.999089
116    5.0  0.998458
117    5.0  0.998506
118    5.0  0.998756
119    5.0  0.998192

[120 rows x 2 columns]


In [8]:
bank_df_classification.to_csv('bank_df_classification')

In [9]:
# Regression Tests
X_train, X_val, y_train, y_val = load_data(dataset = 'bank', task = 'regression')
Ks = [5,10,15]
alphas = [0.1, 0.5]
seeds = range(40,50)

tasks = [(K, alpha, seed) for K in Ks for alpha  in alphas for seed in seeds]

if __name__ == '__main__':
    results = Parallel(n_jobs=-1)(
       delayed(run_experiment)(*t, X_train, y_train, X_val, y_val, task = 'regression', scenario = 1) 
       for t in tqdm(tasks, desc = 'Overall Experiment Progress')
    )
    bank_df_regression = pd.DataFrame(results)

Overall Experiment Progress: 100%|██████████| 60/60 [06:54<00:00,  6.90s/it]


In [10]:
bank_df_regression.to_csv('bank_df_regression')

In [16]:
# Scenario 2
# Classification
X_train_cls, X_val_cls, y_train_cls, y_val_cls = load_data(dataset = 'bank',task='classification')

# Regression
X_train_reg, X_val_reg, y_train_reg, y_val_reg = load_data(dataset = 'bank', task='regression')

seeds = range(40,50)
tasks_cls = [(10, None, seed) for seed in seeds]
tasks_reg = [(10, None, seed) for seed in seeds]

if __name__ == '__main__':
    bank_results_cls = Parallel(n_jobs=-1)(
        delayed(run_experiment)(*t, X_train_cls, y_train_cls, X_val_cls, y_val_cls, 
                                task='classification', scenario=2)
        for t in tqdm(tasks_cls, desc='Scenario 2 Classification')
    )
    
    bank_results_reg = Parallel(n_jobs=-1)(
        delayed(run_experiment)(*t, X_train_reg, y_train_reg, X_val_reg, y_val_reg,
                                task='regression', scenario=2)
        for t in tqdm(tasks_reg, desc='Scenario 2 Regression')
    )

    

Scenario 2 Regression: 100%|██████████| 10/10 [00:00<00:00, 463.62it/s]


In [21]:
bank_results_reg = pd.DataFrame(bank_results_reg)
bank_results_cls = pd.DataFrame(bank_results_cls)

In [22]:
bank_results_reg.to_csv('bank_df_scenario2_reg')
bank_results_cls.to_csv('bank_df_scenario2_class')

In [18]:
bank_df_scenario2 = pd.DataFrame(bank_results_cls + bank_results_reg).drop('alpha', axis = 1)
bank_df_scenario2['task'] = ['classification'] * 10 + ['regression'] * 10
bank_df_scenario2.to_csv('bank_scenario2_results.csv', index=False)